In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch.optim

In [ ]:
# Run device agnostic code
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# Get my csv file
data = pd.read_csv('./asl_landmarks_data.csv')
# print(data.head(4))
print(len(data))

In [ ]:
# Create some data
X = data.drop('label', axis=1).values
y = data['label'].values
# print(X[0])
# print(y[0])
print(len(X))
print(len(y))

In [ ]:
# Label encoding
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(y)
print(f"{y[0]}, {labels[0]}")
print(f"{y[2300]}, {labels[2300]}")

y = labels

print(y[0])
print(y[2300])
print(len(X))

In [ ]:
# Hand Scale Problem

# Reshape to (number_of_samples, 21 landmarks, 3 coordinates)
X_reshaped = X.reshape(-1, 21, 3)

# Get the wrist coordinates (Landmark 0) for every sample
# Shape becomes (number_of_samples, 1, 3)
wrists = X_reshaped[:, 0, :]

# Subtract the wrist from every landmark in that hand
X_normalized = X_reshaped - wrists[:, np.newaxis, :]

# Flatten it back to 63 features
X = X_normalized.reshape(-1, 63)

In [ ]:
# Split to train and test
torch.seed()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(len(X))
print(len(y))
print(len(X_train))
print(len(y_train))
print(len(X_test))
print(len(y_test))

In [ ]:
# Turning the data to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

y_train_tensor = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)

print(f"X_train shape: {X_train_tensor.shape}")
print(f"y_train dtype: {y_train_tensor.shape}")

In [ ]:
# Combining the train and test to one data
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

print(len(train_dataset))

In [ ]:
# Converting to dataloaders
BATCH_SIZE = 32
train_dataloader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataloader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(len(train_dataloader))
print(len(test_dataloader))

In [ ]:
print(X_train.shape[1])
print(y.shape)
features = X_train.shape[1]

num_classes = len(label_encoder.classes_)
print(f"Number of classes: {num_classes}")
print(f"Class names: {label_encoder.classes_}")

In [ ]:
# Building the model

class Sign_Model(nn.Module):
  def __init__(self, input, hidden_layers, output):
    super().__init__()
    self.linear_stack = nn.Sequential(
        nn.Linear(in_features=input, out_features=hidden_layers),
        nn.ReLU(),
        nn.Linear(in_features=hidden_layers , out_features=hidden_layers),
        nn.ReLU(),
        nn.Linear(in_features=hidden_layers , out_features=hidden_layers),
        nn.ReLU(),
        nn.Linear(in_features= hidden_layers, out_features=output)
    )

  def forward(self, x):
    return self.linear_stack(x)

model = Sign_Model(input=features, hidden_layers=8, output=num_classes).to(device)
print(model)

In [ ]:
# Optimizer and loss function

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

In [ ]:
# Train function

def train_step(model, dataloader, loss_fn, optimizer):
  model.train()

  train_loss, train_acc = 0, 0

  for batch, (X, y) in enumerate(dataloader):

    X, y = X.to(device), y.to(device)

    y_pred = model(X)

    loss = loss_fn(y_pred, y)
    train_loss += loss.item()

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
    train_acc += (y_pred_class == y).sum().item() / len(y_pred)

  train_loss = train_loss / len(dataloader)
  train_acc = train_acc / len(dataloader) * 100

  return train_loss, train_acc

In [ ]:
# Test function

def test_step(model, dataloader, loss_fn):
  model.eval()

  test_loss, test_acc = 0, 0

  with torch.inference_mode():

    for batch, (X, y) in enumerate(dataloader):

      X, y = X.to(device), y.to(device)

      y_pred = model(X)

      loss = loss_fn(y_pred, y)
      test_loss += loss.item()

      y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
      test_acc += (y_pred_class == y).sum().item() / len(y_pred)

    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader) * 100

  return test_loss, test_acc

In [ ]:
# Training and testing loops in batches
from tqdm.auto import tqdm
torch.manual_seed(42)

def train(model, train_dataloader, test_dataloader, loss_fn, optimizer, epochs):

  for epoch in tqdm(range(epochs)):

    train_loss, train_acc = train_step(model=model, dataloader=train_dataloader, loss_fn=loss_fn, optimizer=optimizer)
    test_loss, test_acc = test_step(model=model, dataloader=test_dataloader, loss_fn=loss_fn)

    print(f"Epoch: {epoch} || Train Loss: {train_loss:.2f} || Train Accuracy: {train_acc:.2f}% || Test Loss: {test_loss:.2f} || Test Accuracy: {test_acc:.2f}%")

In [ ]:
from timeit import default_timer as timer

start = timer()

train(model=model, train_dataloader=train_dataloader, test_dataloader=test_dataloader, loss_fn=loss_fn, optimizer=optimizer, epochs=30)

end = timer()

print(f'Total training time: {end-start:.3f} seconds')

In [ ]:
# Saving the model
torch.save(obj=model.state_dict(), f='asl_model.pth')

In [ ]:
# From colabs to downloads
# from google.colab import files
# files.download('./asl_model.pth')